In [13]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
#import tushare as ts
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from torch.utils.data import TensorDataset
from tqdm import tqdm
import torch.nn.functional as F
import scipy.io as scio
import time 

In [15]:
import scipy
data = scipy.io.loadmat('real3d.mat')
Dn=data['DataNoisey']

In [16]:
def calculate_snr(clean_data, denoised_data):
    # 确保输入是一维向量，或者直接不带 ord 参数（默认就是 Frobenius 范数，即全图能量）
    norm_clean = np.linalg.norm(clean_data)  # 默认计算所有元素的平方和开根号
    norm_diff = np.linalg.norm(clean_data - denoised_data)
    
    snr = 20 * np.log10(norm_clean / norm_diff)
    return snr

In [17]:
def patch3d(A, l1=4, l2=4, l3=4, s1=2, s2=2, s3=2):
    """
    Optimized patch3d function using stride tricks to improve efficiency
    """
    # Pad the array if necessary to ensure dimensions are divisible by patch sizes
    pad1 = (l1 - A.shape[0] % s1) % s1
    pad2 = (l2 - A.shape[1] % s2) % s2
    pad3 = (l3 - A.shape[2] % s3) % s3
    A_padded = np.pad(A, ((0, pad1), (0, pad2), (0, pad3)), mode='constant')

    # Get new dimensions
    n1, n2, n3 = A_padded.shape
    n1_patches = (n1 - l1) // s1 + 1
    n2_patches = (n2 - l2) // s2 + 1
    n3_patches = (n3 - l3) // s3 + 1

    # Generate the sliding window view
    shape = (n1_patches, n2_patches, n3_patches, l1, l2, l3)
    strides = (s1 * A_padded.strides[0], s2 * A_padded.strides[1], s3 * A_padded.strides[2]) + A_padded.strides
    patches = np.lib.stride_tricks.as_strided(A_padded, shape=shape, strides=strides)

    # Reshape patches to (num_patches, patch_size) format
    patches = patches.reshape(-1, l1 * l2 * l3)
    return patches


def patch3d_inv(X, n1, n2, n3, l1=4, l2=4, l3=4, s1=2, s2=2, s3=2):
    """
    Reconstruct 3D data from 1D patches with optimized inverse patching.

    INPUT
    X: Patches in (num_patches, patch_size) format
    n1, n2, n3: Original 3D data dimensions
    l1, l2, l3: Patch sizes along each dimension
    s1, s2, s3: Shifts between patches along each dimension

    OUTPUT
    A: Reconstructed 3D data
    """

    # Initialize the padded 3D array and mask
    pad1 = (l1 - n1 % s1) % s1
    pad2 = (l2 - n2 % s2) % s2
    pad3 = (l3 - n3 % s3) % s3
    A = np.zeros((n1 + pad1, n2 + pad2, n3 + pad3))
    mask = np.zeros_like(A)

    # Reshape 1D patches to the 3D patch size for easier handling
    X = X.reshape(-1, l1, l2, l3)

    # Calculate the number of patches along each dimension
    n1_patches = (A.shape[0] - l1) // s1 + 1
    n2_patches = (A.shape[1] - l2) // s2 + 1
    n3_patches = (A.shape[2] - l3) // s3 + 1

    # Iterate over each patch and place it into the appropriate location
    idx = 0
    for i in range(n1_patches):
        for j in range(n2_patches):
            for k in range(n3_patches):
                # Place the current patch in the reconstruction array and update mask
                A[i * s1:i * s1 + l1, j * s2:j * s2 + l2, k * s3:k * s3 + l3] += X[idx]
                mask[i * s1:i * s1 + l1, j * s2:j * s2 + l2, k * s3:k * s3 + l3] += 1
                idx += 1

    # Avoid division by zero by replacing zeros with ones in the mask before division
    mask[mask == 0] = 1
    A /= mask

    # Trim padding to match the original dimensions
    return A[:n1, :n2, :n3]

In [18]:
w1 = 12
w2 = 12
w3 = 12
z1 = 4
z2 = 2
z3 = 2

dataInput = patch3d(Dn,w1,w2,w3,z1,z2,z3)
#dataInputP = np.transpose(dataInputP)
#dataPatches = np.reshape(dataInput,(dataInput.shape[0],w1*w2*w3))
dataInput = dataInput.astype(np.float32)
Pdata = torch.from_numpy(dataInput)
print(Pdata.dtype)
dataPatches = np.array(dataInput)
print(dataPatches.shape)

torch.float32
(31806, 1728)


In [19]:
# 数据分割：80% 训练数据，20% 验证数据
train_size = int(len(Pdata) * 0.8)
train = Pdata[:train_size]
valid = Pdata[train_size:]

# 打印训练集和验证集的形状
print(f"Train shape: {train.shape}")
print(f"Valid shape: {valid.shape}")
#创建 TensorDataset
train_data = TensorDataset(train)
valid_data = TensorDataset(valid)

# 设置 batch_size
batch_size1 = 64


# 创建 DataLoader
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size1, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size1, shuffle=True)

Train shape: torch.Size([25444, 1728])
Valid shape: torch.Size([6362, 1728])


In [20]:
#定义Fully connected (FC) block
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU()
        self.bn = nn.BatchNorm1d(output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x) 
        x = self.bn(x)
        x = self.dropout(x)
        
        return x

In [21]:
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.1):
        super().__init__()


        self.fcb0 = FCB(input_size, 128, dropout)
        self.fcb1 = FCB(128, 64, dropout)

        self.fcb2 = FCB(64, 32, dropout)

        self.fcb3 = FCB(32, 16, dropout)

        self.fcb4 = FCB(16, 8, dropout)

        self.fcb5 = FCB(8, 4, dropout)

        self.fcb6 = FCB(4, 4, dropout)

        


    def forward(self, x):

        #x = x.reshape(x.shape[0],1,x.shape[1])

        x1 = self.fcb0(x)#x->128
        x2 = self.fcb1(x1)

        x3 = self.fcb2(x2)

        x4 = self.fcb3(x3)

        x5 = self.fcb4(x4)

        x6 = self.fcb5(x5)

        x7 = self.fcb6(x6)

        x8 = self.fcb6(x7)
        

        

        return x2,x3,x4,x5,x6,x7,x8

In [22]:
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.1):
        super().__init__()
        self.pab1 = FCB(8, 8, dropout)
        self.pab2 = FCB(16, 16, dropout)
        self.pab3 = FCB(32, 32, dropout)
        self.pab4 = FCB(64, 64, dropout)
        self.pab5 = FCB(128, 128, dropout)
        self.pab6 = FCB(128, output_size, dropout)

        self.activation = nn.Identity()


    def forward(self,x2, x3,x4,x5,x6,x7,x8):
        
        x = torch.cat((x6,x8),dim=1)
        x = self.pab1(x)
        x = torch.cat((x,x5),dim=1)
        x = self.pab2(x)
        x = torch.cat((x,x4),dim=1)
        x = self.pab3(x)
        x = torch.cat((x,x3),dim=1)
        x = self.pab4(x)
        x = torch.cat((x,x2),dim=1)
        x = self.pab5(x)
        x = self.pab6(x)

        x = self.activation(x)
        return x

In [23]:
class AutoEncoder(nn.Module):
    def __init__(self,input_size, output_size):
        super().__init__()
        self.encoder = Encoder(input_size)
        self.decoder = Decoder(output_size)

    def forward(self, x):
        x2, x3,x4,x5,x6,x7,x8= self.encoder(x)
        x = self.decoder(x2, x3,x4,x5,x6,x7,x8)

        return x

In [24]:
device = torch.device("cuda") 
model = AutoEncoder(w1*w2*w3,w1*w2*w3).to(device)
#将模型转移到GPU上
#criterion = MeanHuberLoss(delta=1.3)
#criterion = WelschLoss(delta=0.5)
#criterion = Loss0(delta=0.46,r=0.05)#0.5 and 0.2,SNR:-8dB ok
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

In [30]:
start_time = time.time()
es_cnt = 0
es_thres = 5
prev_valid_loss = float('inf')
loss_train = []
loss_valid = []
num_epochs = 100 # 总训练轮数

for epoch in range(num_epochs):
    # 训练
    model.train()
    train_loss = 0.0
    for i, (batch) in enumerate(train_loader):
        # 数据转到 device
        train_batch = batch[0].to(device)
        
        # 训练步骤
        optimizer.zero_grad()
        outputs = model(train_batch)
        loss = criterion(outputs, train_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # 计算平均训练损失
    train_loss = train_loss / (np.ceil(train.size(0) / batch_size1))
    loss_train.append(train_loss)
    
    # 验证
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for i, (batch) in enumerate(valid_loader):
            val_batch = batch[0].to(device)
            outputs = model(val_batch)
            loss = criterion(outputs, val_batch)
            valid_loss += loss.item()
    
    # 计算平均验证损失
    valid_loss = valid_loss / (np.ceil(valid.size(0) / batch_size1))
    loss_valid.append(valid_loss)
    
    # 打印当前 epoch 的损失
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")
    
    # Early stopping
    if valid_loss >= prev_valid_loss:
        es_cnt += 1
    else:
        es_cnt = 0

    if es_cnt >= es_thres:
        print(f"Early stopped at epoch {epoch}, validation loss stop improving")
        break

    prev_valid_loss = valid_loss

# 打印最终损失记录
print('Final Train Loss: ', loss_train)
print('Final Valid Loss: ', loss_valid)

# 打印总训练时间
current_time = time.time()
time_sum = current_time - start_time
time_sum_minutes = time_sum / 60
print(f"Total training time: {time_sum_minutes:.2f} minutes")

Epoch [1/100], Train Loss: 0.2534, Valid Loss: 0.2579
Epoch [2/100], Train Loss: 0.2521, Valid Loss: 0.2563
Epoch [3/100], Train Loss: 0.2509, Valid Loss: 0.2530
Epoch [4/100], Train Loss: 0.2500, Valid Loss: 0.2514
Epoch [5/100], Train Loss: 0.2491, Valid Loss: 0.2498
Epoch [6/100], Train Loss: 0.2484, Valid Loss: 0.2485
Epoch [7/100], Train Loss: 0.2478, Valid Loss: 0.2486
Epoch [8/100], Train Loss: 0.2473, Valid Loss: 0.2475
Epoch [9/100], Train Loss: 0.2468, Valid Loss: 0.2474
Epoch [10/100], Train Loss: 0.2464, Valid Loss: 0.2471
Epoch [11/100], Train Loss: 0.2460, Valid Loss: 0.2459
Epoch [12/100], Train Loss: 0.2453, Valid Loss: 0.2448
Epoch [13/100], Train Loss: 0.2447, Valid Loss: 0.2442
Epoch [14/100], Train Loss: 0.2443, Valid Loss: 0.2439
Epoch [15/100], Train Loss: 0.2439, Valid Loss: 0.2440
Epoch [16/100], Train Loss: 0.2435, Valid Loss: 0.2435
Epoch [17/100], Train Loss: 0.2434, Valid Loss: 0.2434
Epoch [18/100], Train Loss: 0.2432, Valid Loss: 0.2439
Epoch [19/100], Tra

In [31]:
loss_train = pd.DataFrame(loss_train)
loss_valid = pd.DataFrame(loss_valid)
loss = pd.concat([loss_train,loss_valid],axis=1)

In [32]:
loss.columns = ['train_loss','valid_loss']

In [33]:
torch.save(model.state_dict(), r'.\model_3dreal2patch.pth')

In [35]:
model = AutoEncoder(w1*w2*w3,w1*w2*w3).to(device)
Pdata = Pdata.to(device)
model.load_state_dict(torch.load(r'.\model_3dreal2patch.pth'))
model.eval()
with torch.no_grad():
    output = model(Pdata)
    loss = criterion(output, Pdata)
output = output.cpu()
output = output.numpy()
# ========== 新增：b保存去噪结果为mat形式（反分块） ==========
# 重构去噪后的数据
n1 = Dn.shape[0]  # 对应MATLAB：n1 = size(Dn, 1)
n2 = Dn.shape[1]  # 对应MATLAB：n2 = size(Dn, 2)
n3 = Dn.shape[2]  # 对应MATLAB：n3 = size(Dn, 3)
D_denoised = patch3d_inv(output,n1,n2,n3,w1,w2,w3,z1,z2,z3)
scipy.io.savemat(r"output_3dreal2patch.mat", 
        {'D_denoised': D_denoised})  # 'D_denoised'是MAT文件里的变量名

In [36]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()